# Retrieval-Augmented Generation (RAG)

In [1]:
import chromadb
import dotenv
from pathlib import Path
from agents import Agent, Runner, function_tool, trace

dotenv.load_dotenv()

True

Create a static calorie table that we can use as a tool:

In [2]:
# We populated the RAG with the data from the data/calories.csv file in
# the rag_setup.ipynb notebook
chroma_client = chromadb.PersistentClient("../chroma")
nutrition_db = chroma_client.get_collection(name="nutrition_db")
nutrition_qna = chroma_client.get_collection(name="nutrition_qna")

In [3]:
results = nutrition_db.query(query_texts=["banana"], n_results=2)
for i, doc in enumerate(results["documents"][0]):
    print(sorted(results["metadatas"][0][i].items()))
    print(doc)
    print("\n")

[('calories_per_100g', 89.0), ('food_category', 'fruits'), ('food_item', 'banana'), ('keywords', 'banana_fruits'), ('kj_per_100g', 374.0), ('serving_info', '100g')]
Food: Banana
        Category: Fruits
        Nutritional Information:
        - Calories: 89 per 100g
        - Energy: 374 kJ per 100g
        - Serving size reference: 100g

        This is a fruits food item that provides 89 calories per 100 grams.


[('calories_per_100g', 50.0), ('food_category', '(fruit)juices'), ('food_item', 'banana juice'), ('keywords', 'banana_juice_(fruit)juices'), ('kj_per_100g', 210.0), ('serving_info', '100ml')]
Food: Banana Juice
        Category: (Fruit)Juices
        Nutritional Information:
        - Calories: 50 per 100g
        - Energy: 210 kJ per 100g
        - Serving size reference: 100ml

        This is a (fruit)juices food item that provides 50 calories per 100 grams.




In [4]:
results = nutrition_qna.query(query_texts=["What food should I avoid during first trimestsr"], n_results=2)
for i, doc in enumerate(results["documents"][0]):
    print(sorted(results["metadatas"][0][i].items()))
    print(doc)
    print("\n")

[('is_pregnancy', True)]
Question: What kind of food should be given priority during the last phase of pregnancy?
        Answer: In the third trimester (weeks 28-40), a diet rich in nutrients is vital to support both the baby's development and the mother's body preparing for childbirth. The focus should be on foods that promote eye formation, bone growth, organ maturation, brain expansion, and lung improvement.

        This Q&A pair provides information about nutrition and health topics.


[('is_pregnancy', False)]
Question: What are some suitable first foods for children ranging from four to twelve months old?
        Answer: Some examples of complementary foods recommended for infants aged between 4 to 12 months include Fruit Juices, Green Leafy Vegetables, Cereals, Egg yolk, Starchy vegetables and fruits, Pulses, Vegetables and pulses, Whole egg, Meat, vegetables, and fruits, and soups in milk.

        This Q&A pair provides information about nutrition and health topics.




In [5]:
@function_tool
def calorie_lookup_tool(query: str, max_results: int = 3) -> str:
    """
    Tool function for a RAG database to look up calorie information for specific food items, but not for meals.

    Args:
        query: The food item to look up.
        max_results: The maximum number of results to return.

    Returns:
        A string containing the nutrition information.
    """

    results = nutrition_db.query(query_texts=[query], n_results=max_results)

    if not results["documents"][0]:
        return f"No nutrition information found for: {query}"

    # Format results for the agent
    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        food_item = metadata["food_item"].title()
        calories = metadata["calories_per_100g"]
        category = metadata["food_category"].title()

        formatted_results.append(
            f"{food_item} ({category}): {calories} calories per 100g"
        )

    return "Nutrition Information:\n" + "\n".join(formatted_results)

In [26]:
@function_tool

def nutrion_qa_tool(query: str, max_results: int = 3) -> str:
    """
    Tool function for a RAG database that contains common Q&A about nutrition in general with emphasis on malnutrition and nutrition during pregnancy.

    Args:
        query: The food item to look up.
        max_results: The maximum number of results to return.

    Returns:
        A string containing the nutrition information.
    """

    results = nutrition_qna.query(query_texts=[query], n_results=max_results)

    if not results["documents"][0]:
        return f"No nutrition information found for: {query}"

   # Format results for the agent
    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        formatted_results.append(
            f"{doc}"
        )

        

    return "Retrieved information on nutrition:\n" + "\n".join(formatted_results)

In [27]:
nutrion_qa_tool("Can pregnant women eat fish?")

TypeError: 'FunctionTool' object is not callable

Let's test this out: 

_The following cell only works before you add the `@function_tool` annotation to `calorie_lookup_tool` function_

In [28]:
calorie_lookup_tool('bananas')

TypeError: 'FunctionTool' object is not callable

In [30]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information.
    You give concise answers.
    If you need to look up calorie information, use the calorie_lookup_tool.
    If the user asks general questions on nutrition use nutrition_qa_tool.
    """,
    tools=[calorie_lookup_tool, nutrion_qa_tool]
)

In [29]:
with trace("Nutrition Assistant with RAG"):
    result = await Runner.run(
        calorie_agent,
        "How many calories are in total in a banana and an apple? Also give calories per 100g",
    )
    print(result.final_output)

- Typical calories for one medium banana (≈118 g): ~105 kcal
- Typical calories for one medium apple (≈182 g): ~95 kcal
- Total for one banana + one apple: ~200 kcal

Calories per 100 g:
- Banana: 89 kcal per 100 g
- Apple: 52 kcal per 100 g


In [31]:
with trace("Nutrition Assistant with RAG"):
    result = await Runner.run(
        calorie_agent,
        "Can pregnant women eat fish?",
    )
    print(result.final_output)

Yes, most fish is safe and beneficial during pregnancy, but follow these tips:

- Choose low-mercury options: salmon, sardines, trout, tilapia, shrimp, and tuna (light canned) are good choices.
- Limit high-mercury fish: avoid shark, swordfish, king mackerel, and tilefish; limit albacore tuna.
- Cook fish thoroughly to avoid pathogens.
- Aim about 2–3 servings (8–12 ounces total) of low-mercury fish per week.
- Avoid raw or undercooked fish and raw shellfish.

If you have specific health concerns or dietary restrictions, consult your healthcare provider.
